In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 2: Data Cleaning -> olist_products_dataset.csv (for Tableau dashboard)
==================================================================================
 Purpose : Clean the product catalogue table for use as a Tableau dimension
           table joined to olist_order_items_dataset.csv via 'product_id'.

 IMPORTANT DIFFERENCE FROM THE PRIMARY (ML) DATASET:
     For a BI/dashboard dataset, rows must NEVER be dropped just because a
     value is missing. 'product_id' is a foreign key referenced by the
     order_items fact table -> dropping a product row here would create an
     orphaned key in Tableau, causing that product's orders to silently
     disappear (or break) in any join/visualization. So missing values are
     filled with clearly-labeled placeholders instead of removed.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_products_dataset.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/olist_products_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. DUPLICATE RECORD REMOVAL
# ----------------------------------------------------------------------------------
section("2. DUPLICATE RECORD REMOVAL")

full_dupes = df.duplicated().sum()
id_dupes = df["product_id"].duplicated().sum()
print(f"Fully duplicated rows before cleaning : {full_dupes}")
print(f"Duplicated product_id before cleaning : {id_dupes}")

before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]
print(f"Rows removed as duplicates            : {before_rows - after_rows}")
print("Note: duplicate 'product_id' rows are NOT removed by ID alone here, since")
print("doing so could arbitrarily discard a legitimate product record. None were")
print("found in this dataset, so this is a safety check only, not an active step.")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE TREATMENT (placeholder-fill, never drop rows)
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE TREATMENT")

missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
print("Missing values before cleaning:")
print(missing_before)

# 3.1 Categorical: fill with an explicit, Tableau-friendly placeholder label
df["product_category_name"] = df["product_category_name"].fillna("unknown_category")

# 3.2 Descriptive numeric fields (name/description length, photo count):
#     fill with 0, since a missing value here genuinely means "not provided"
descriptive_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
for col in descriptive_cols:
    df[col] = df[col].fillna(0)

# 3.3 Physical dimension fields (weight/size): only 2 rows affected.
#     Fill with the median, since a Tableau size/weight chart should not show
#     a blank or zero for a real physical product.
dimension_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dimension_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  Filled '{col}' missing values with median = {median_val}")

missing_after = df.isnull().sum().sum()
print(f"\nTotal missing values after cleaning: {missing_after}")


# ----------------------------------------------------------------------------------
# 4. DATA TYPE CORRECTIONS
# ----------------------------------------------------------------------------------
section("4. DATA TYPE CORRECTIONS")

int_cols = descriptive_cols + dimension_cols
for col in int_cols:
    df[col] = df[col].astype(int)

print(df.dtypes)


# ----------------------------------------------------------------------------------
# 5. CATEGORY LABEL CONSISTENCY CHECK
# ----------------------------------------------------------------------------------
section("5. CATEGORY LABEL CONSISTENCY CHECK")

print(f"Total categories (including placeholder): {df['product_category_name'].nunique()}")
print("Note: category names are in Portuguese (source dataset). If your Tableau")
print("dashboard needs English labels, join this column against Olist's official")
print("'product_category_name_translation.csv' file rather than editing values here.")


# ----------------------------------------------------------------------------------
# 6. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("6. FINAL VALIDATION")

print(f"Final shape              : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining missing values : {df.isnull().sum().sum()}")
print(f"Remaining duplicate rows : {df.duplicated().sum()}")
print(f"product_id count matches original row count "
      f"(no rows lost) : {df.shape[0] == before_rows}")


# ----------------------------------------------------------------------------------
# 7. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("7. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("PRODUCTS DATASET CLEANING COMPLETE")
print("This file is now safe to load into Tableau and join to order_items on 'product_id'")
print("with no orphaned keys and no blank category/dimension fields in visuals.")


 1. LOAD RAW DATA
Raw dataset loaded: 32951 rows, 9 columns

 2. DUPLICATE RECORD REMOVAL
Fully duplicated rows before cleaning : 0
Duplicated product_id before cleaning : 0
Rows removed as duplicates            : 0
Note: duplicate 'product_id' rows are NOT removed by ID alone here, since
doing so could arbitrarily discard a legitimate product record. None were
found in this dataset, so this is a safety check only, not an active step.

 3. MISSING VALUE TREATMENT
Missing values before cleaning:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
  Filled 'product_weight_g' missing values with median = 700.0
  Filled 'product_length_cm' missing values with median = 25.0
  Filled 'product_height_cm' missing values with median = 13.0
  Filled 'product_width_c